# 🩺 Diabetes Prediction

This notebook presents a complete machine learning pipeline developed for competition on health data. The goal is to predict the probability of a diabetes diagnosis from a set of patient health indicators, using a synthetically generated dataset derived from the Diabetes Health Indicators Dataset.

The work is structured to cover the full ML workflow: exploratory analysis, preprocessing, feature engineering, model training and comparison, evaluation and final prediction.

**Source competition**: https://www.kaggle.com/competitions/playground-series-s5e12

**Author**: Sansone Lorenzo — University of Bologna, Academic Year 2025/2026

## 🗺️ Notebook Structure

1. Introduction
2. Setup
3. Exploratory Data Analysis (EDA)
4. Preprocessing & Feature Engineering
5. Baseline Model
6. Model Training & Comparison
7. Evaluation
8. Model Selection & Ensemble
9. Prediction & Submission
10. Conclusions

## 📌 **1. Introduction**

This notebook addresses a binary classification problem: **predicting the probability that a patient has been diagnosed with diabetes**, based on a set of health indicators.

The dataset was synthetically generated from a deep learning model trained on the [Diabetes Health Indicators Dataset](https://www.kaggle.com/datasets/mohankrishnathalla/diabetes-health-indicators-dataset/data), meaning feature distributions are close — but not identical — to the original.

---

### 🎯 1.1 Objective
The goal of this competition is to leverage machine learning techniques to predict the probability that a patient will be diagnosed with diabetes based on various clinical and demographic features.

Predict the **probability** of `diagnosed_diabetes` for each sample in the test set.  
The primary evaluation metric is **ROC-AUC** (Area Under the Receiver Operating Characteristic Curve).

---

### 📂 1.2 Files
| File | Description |
|---|---|
| `train.csv` | Training set with features and target label |
| `test.csv` | Test set — features only, no labels |
| `sample_submission.csv` | Expected output format for predictions |


## **2. Setup**

### 2.1 — Install Missing Dependencies (if needed)

The cell below installs any libraries not available by default in the environment.  
If you're running this on Kaggle or Google Colab, most of these are pre-installed.

In [1]:
# Install libraries not available by default
# Uncomment as needed depending on the environment

# !pip install numpy
# !pip install -U scikit-learn
# !pip install pandas

### 2.2 — Import Libraries

In [2]:
# ── Core ──────────────────────────────────────────────────────────────────────
import numpy as np                          # Numerical operations
import pandas as pd                         # Data manipulation
import warnings                             # Suppress non-critical warnings

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt             # Base plotting library
import matplotlib.ticker as mticker
import seaborn as sns                       # Statistical visualizations

# ── Utilities ─────────────────────────────────────────────────────────────────
import os
import time
from IPython.display import display

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


### 2.3 — Global Configuration

Centralizing configuration constants makes experiments reproducible and easy to tweak.
- `RANDOM_STATE`: fixes all stochastic operations so results are reproducible
- `N_FOLDS`: number of folds for cross-validation
- `TEST_SIZE`: fraction of training data used for the local validation set
- `TARGET`: name of the column we want to predict

In [3]:
# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_STATE = 42           # Fixed seed for all random operations
np.random.seed(RANDOM_STATE)

# ── Cross-Validation ──────────────────────────────────────────────────────────
N_FOLDS = 5                 # Number of StratifiedKFold folds

# ── Dataset ───────────────────────────────────────────────────────────────────
TARGET = 'diagnosed_diabetes'   # Name of the target column
DATA_DIR = '/kaggle/input/competitions/playground-series-s5e12' #Adjust according the dataset location 

print(f"✅ Configuration set:")
print(f"   Random state : {RANDOM_STATE}")
print(f"   CV folds     : {N_FOLDS}")
print(f"   Target col   : '{TARGET}'")

✅ Configuration set:
   Random state : 42
   CV folds     : 5
   Target col   : 'diagnosed_diabetes'


### 2.4 — Load the Data

We load all three CSV files provided by the competition:
- `train.csv`: used to train and validate our models
- `test.csv`: the set for which we generate predictions
- `sample_submission.csv`: shows us the exact format expected for submission

In [4]:
# ── Load Datasets ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_submission_df = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print("✅ Files loaded successfully.")
print(f"\n{'Dataset':<22} {'Rows':>8} {'Columns':>10}")
print("-" * 42)
print(f"{'train.csv':<22} {train_df.shape[0]:>8} {train_df.shape[1]:>10}")
print(f"{'test.csv':<22} {test_df.shape[0]:>8} {test_df.shape[1]:>10}")
print(f"{'sample_submission.csv':<22} {sample_submission_df.shape[0]:>8} {sample_submission_df.shape[1]:>10}")

✅ Files loaded successfully.

Dataset                    Rows    Columns
------------------------------------------
train.csv                700000         26
test.csv                 300000         25
sample_submission.csv    300000          2
